<a href="https://colab.research.google.com/github/Dappa-77/mineral-hardness-classifier-with-CNN/blob/main/mohrs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving MohrsSplit.zip to MohrsSplit.zip


In [ ]:
import zipfile

with zipfile.ZipFile("MohrsSplit.zip", "r") as zip_ref:
    zip_ref.extractall("dataset")

print("✅ Unzipped successfully!")

✅ Unzipped successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import shutil
import os

def remove_checkpoints(root_dir):
    for path, dirs, files in os.walk(root_dir):
        for d in dirs:
            if d == ".ipynb_checkpoints":
                full_path = os.path.join(path, d)
                shutil.rmtree(full_path)
                print(f"Removed: {full_path}")

remove_checkpoints("dataset")

Removed: dataset/MohrsSplit/val/.ipynb_checkpoints
Removed: dataset/MohrsSplit/train/.ipynb_checkpoints
Removed: dataset/MohrsSplit/test/.ipynb_checkpoints


In [ ]:
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
import torch

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),

    # 🔥 Augmentation (only for training)
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3
    ),
    transforms.RandomResizedCrop(224),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3),

    transforms.ToTensor()
])

In [ ]:
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
train_dir = "dataset/MohrsSplit/train"

In [ ]:
val_dir = "dataset/MohrsSplit/train"

In [ ]:
train_data = datasets.ImageFolder(train_dir, transform=train_transforms)

In [ ]:
val_data = datasets.ImageFolder(
    val_dir,
    transform=val_transforms
)

In [ ]:
img, label = train_data[22]

In [ ]:
print(train_data.classes)

['apatite_mineral_rock', 'calcite_mineral_specimen', 'corundum_mineral_ruby_sapphire', 'diamond_rough_crystal', 'fluorite_crystal_mineral', 'gypsum_mineral_specimen', 'orthoclase_feldspar_mineral', 'quartz_crystal_mineral', 'talc_mineral_specimen', 'topaz_crystal_stone']


In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
from collections import Counter

labels = [label for _, label in train_data]

print(Counter(labels))

Counter({3: 27, 0: 22, 2: 22, 1: 19, 8: 19, 5: 18, 6: 16, 9: 15, 4: 11, 7: 7})


In [ ]:
from torchvision import models
import torch.nn as nn

model = models.resnet50(pretrained=True)

# Replace final layer for 10 minerals
model.fc = nn.Linear(2048, 10)

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 30

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss:.4f} | Accuracy: {train_acc:.2f}%")

Epoch [1/30] Loss: 9.9131 | Accuracy: 39.20%
Epoch [2/30] Loss: 9.8508 | Accuracy: 46.59%
Epoch [3/30] Loss: 9.2241 | Accuracy: 48.30%
Epoch [4/30] Loss: 9.2215 | Accuracy: 51.14%
Epoch [5/30] Loss: 8.1975 | Accuracy: 53.98%
Epoch [6/30] Loss: 8.3656 | Accuracy: 48.86%
Epoch [7/30] Loss: 8.3070 | Accuracy: 52.84%
Epoch [8/30] Loss: 8.4522 | Accuracy: 50.57%
Epoch [9/30] Loss: 8.5757 | Accuracy: 50.57%
Epoch [10/30] Loss: 8.0250 | Accuracy: 50.00%
Epoch [11/30] Loss: 8.0225 | Accuracy: 53.98%
Epoch [12/30] Loss: 6.9119 | Accuracy: 60.80%
Epoch [13/30] Loss: 7.9477 | Accuracy: 53.98%
Epoch [14/30] Loss: 6.9222 | Accuracy: 62.50%
Epoch [15/30] Loss: 7.1971 | Accuracy: 59.66%
Epoch [16/30] Loss: 7.2282 | Accuracy: 60.23%
Epoch [17/30] Loss: 7.1228 | Accuracy: 55.68%
Epoch [18/30] Loss: 6.7492 | Accuracy: 63.64%
Epoch [19/30] Loss: 7.7443 | Accuracy: 59.09%
Epoch [20/30] Loss: 6.4633 | Accuracy: 67.05%
Epoch [21/30] Loss: 6.7056 | Accuracy: 64.77%
Epoch [22/30] Loss: 5.9047 | Accuracy: 66.4